In [ ]:
import cv2
import numpy as np
import tqdm.notebook as tqdm
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
imgs = []

# Load the video
video_path = Path('dropbox') / 'checkerboard_25mm_8x6.MOV'
cap = cv2.VideoCapture(video_path.as_posix())

# Parameters
frame_skip = 30  # number of frames to skip
frame_count = 0
with tqdm.tqdm(total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))) as pbar:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Save frame every 'frame_skip' frames
        if frame_count % frame_skip == 0:
            imgs.append(frame)

        frame_count += 1
        pbar.update(1)

cap.release()

In [ ]:

# Termination criteria for refining the detected corners
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# Prepare object points: (0,0,0), (1,0,0), (2,0,0), ..., (7,5,0)
objp = np.zeros((6*8,3), np.float32)
objp[:,:2] = np.mgrid[0:8,0:6].T.reshape(-1,2) * 0.025  # multiply by square size

# Arrays to store object points and image points
objpoints = []  # 3d points in real world space
imgpoints = []  # 2d points in image plane

# Load all images
for img in tqdm.tqdm(imgs[::6]):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Find the chess board corners
    ret, corners = cv2.findChessboardCorners(gray, (8,6), None)

    # If found, add object points, image points
    if ret:
        objpoints.append(objp)
        corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
        imgpoints.append(corners)

        # Draw and display the corners
        cv2.drawChessboardCorners(img, (8,6), corners2, ret)
        plt.figure(figsize=(6, 4))
        plt.imshow(img, cmap='gray')
        cv2.waitKey(500)
    else:
        print("Corners not found in one image")

cv2.destroyAllWindows()

# Camera calibration
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

# Save the camera matrix and distortion coefficients
np.savez('data/camera_calibration.npz', mtx=mtx, dist=dist, rvecs=rvecs, tvecs=tvecs)

print("Calibration is completed.")
